# 🎵 Spotify Recommendation System — 5 Models

| # | Model | Task |
|---|---|---|
| 1 | **KNN** | Recommend songs by audio similarity |
| 2 | **K-Means** | Cluster songs by sound profile |
| 3 | **Random Forest** | Predict popularity score |
| 4 | **Linear Regression** | Baseline popularity prediction |
| 5 | **DBSCAN** | Density-based song grouping |

**Dataset facts (verified from real file):**
- 170,653 songs × 19 columns, years 1921–2020
- No null values in any column
- 12,968 duplicate (name+artists) pairs → deduplicated
- `artists` stored as string-encoded list → parsed with `ast.literal_eval`
- 27,892 songs with popularity = 0 (obscure tracks)
- 143 songs with tempo = 0, 177 under 30s, 110 over 30min → dropped
- `loudness` [-60, +3.8] and `tempo` [0, 243] → need normalization


## ⚙️ Step 1: Install & Import

In [ ]:
!pip install -q scikit-learn joblib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, mean_squared_error,
    mean_absolute_error, r2_score
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120
print("✅ All libraries imported")

## 📥 Step 2: Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload data.csv
print("✅ Uploaded:", list(uploaded.keys()))

## 🔧 Step 3: Preprocessing
> Every decision is based on the actual data audit — not assumptions.

In [ ]:
df_raw = pd.read_csv("data.csv")
print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns  : {df_raw.columns.tolist()}")
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# ── 1. Parse artists: stored as "['Ed Sheeran', 'Bieber']" string ──
def parse_artists(val):
    try:
        return ", ".join(ast.literal_eval(val))
    except Exception:
        return str(val).strip("[]'\"")

df["artists_clean"] = df["artists"].apply(parse_artists)

# ── 2. Dedup: keep highest popularity version of each (name+artists) ──
before = len(df)
df = df.sort_values("popularity", ascending=False)
df = df.drop_duplicates(subset=["name", "artists"], keep="first")
print(f"Dropped {before - len(df):,} duplicate (name+artists) rows")

# ── 3. Drop bad audio rows (verified from audit) ──────────────
before = len(df)
df = df[df["tempo"]        >  0      ]   # 143 rows: Spotify couldn't detect BPM
df = df[df["duration_ms"]  >= 30000  ]   # 177 rows: intros/skits < 30s
df = df[df["duration_ms"]  <= 1800000]   # 110 rows: live sets > 30min
df = df[df["loudness"]     >= -40    ]   #  91 rows: silence / bad recording
print(f"Dropped {before - len(df):,} bad audio rows")

# ── 4. Add helper columns ─────────────────────────────────────
df["is_obscure"]   = (df["popularity"] == 0).astype(int)
df["duration_min"] = (df["duration_ms"] / 60000).round(2)
df["explicit"]     = df["explicit"].astype(bool)

df = df.reset_index(drop=True)
print(f"\n✅ Clean dataset: {len(df):,} rows")

In [ ]:
# ── Audio features — verified column names from actual file ───
AUDIO_FEATURES = [
    "acousticness",     # [0, 1]
    "danceability",     # [0, 1]
    "energy",           # [0, 1]
    "instrumentalness", # [0, 1]
    "liveness",         # [0, 1]
    "loudness",         # [-60, +3.8]  ← must normalize
    "speechiness",      # [0, 1]
    "tempo",            # [0, 243]     ← must normalize
    "valence",          # [0, 1]
]

TARGET = "popularity"   # int [0, 100] — target for regression models

# ── Normalize to [0, 1] ───────────────────────────────────────
scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[AUDIO_FEATURES] = scaler.fit_transform(df[AUDIO_FEATURES])

print("Normalization check (all should be 0.0 min, 1.0 max):")
display(df_scaled[AUDIO_FEATURES].agg(["min","max"]).round(4))

## 📊 Step 4: EDA

In [ ]:
# Correlation heatmap
plt.figure(figsize=(11, 8))
corr = df[AUDIO_FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation — Audio Features vs Popularity", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, feat, color in zip(axes.flatten(), AUDIO_FEATURES,
                           sns.color_palette("tab10", 9)):
    sns.histplot(df[feat], bins=50, kde=True, ax=ax, color=color)
    ax.set_title(feat)
    ax.set_xlabel("")
plt.suptitle("Audio Feature Distributions (raw)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Popularity distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[TARGET], bins=50, kde=True, ax=ax1, color="steelblue")
ax1.set_title("Popularity — All (incl. zeros)")
sns.histplot(df[df[TARGET] > 0][TARGET], bins=50, kde=True,
             ax=ax2, color="coral")
ax2.set_title("Popularity — Excluding Zeros")
plt.tight_layout()
plt.show()
print(f"popularity=0 : {(df[TARGET]==0).sum():,} songs")
print(f"popularity>0 : {(df[TARGET]>0).sum():,} songs")

In [ ]:
# Audio feature trends over time
yearly = df.groupby("year")[AUDIO_FEATURES + [TARGET]].median()
feats_to_plot = ["popularity", "danceability", "energy", "acousticness", "valence"]

fig, axes = plt.subplots(len(feats_to_plot), 1, figsize=(14, 14), sharex=True)
for ax, feat in zip(axes, feats_to_plot):
    ax.plot(yearly.index, yearly[feat], linewidth=2, color="steelblue")
    ax.fill_between(yearly.index, yearly[feat], alpha=0.15)
    ax.set_ylabel(feat)
axes[-1].set_xlabel("Year")
plt.suptitle("Feature Trends 1921–2020 (yearly median)", fontsize=14)
plt.tight_layout()
plt.show()

---
# 🤖 THE 5 MODELS
---

## 🎧 Model 1 — KNN (Content-Based Recommendation)
**Task:** Given a song, find the N most sonically similar songs.

**How it works:** Computes cosine distance between normalized audio feature vectors. Songs with the smallest cosine distance are most similar.

**Input →** audio feature vector of a query song  
**Output →** ranked list of similar songs with similarity score

In [ ]:
print(f"Training KNN on {len(df_scaled):,} songs...")

X_knn = df_scaled[AUDIO_FEATURES].values

knn_model = NearestNeighbors(
    n_neighbors=11,    # 10 recs + 1 (query song itself)
    metric="cosine",
    algorithm="brute", # required for cosine
    n_jobs=-1
)
knn_model.fit(X_knn)
print("✅ KNN trained!")

In [ ]:
def recommend_knn(song_name, df_scaled, knn_model, n=10, popular_only=False):
    """
    Input  : song name (partial or full, case-insensitive)
    Output : DataFrame — top-N similar songs with similarity score
    """
    matches = df_scaled[
        df_scaled["name"].str.lower().str.contains(song_name.lower(), na=False)
    ]
    if matches.empty:
        print(f"❌ '{song_name}' not found."); return None

    song = matches.iloc[0]
    print(f"\n🎧 Query  : '{song['name']}'")
    print(f"   Artist : {song['artists_clean']}")
    print(f"   Year   : {song['year']}  |  Popularity: {song['popularity']}")
    print("-" * 55)

    query = song[AUDIO_FEATURES].values.reshape(1, -1)
    distances, indices = knn_model.kneighbors(query)

    results = df_scaled.iloc[indices[0][1:]].copy()
    results["similarity_score"] = (1 - distances[0][1:]).round(4)

    if popular_only:
        results = results[results["popularity"] > 0]

    return results.head(n)[
        ["name", "artists_clean", "year", "popularity", "similarity_score"]
    ].reset_index(drop=True)


recs_knn = recommend_knn("Shape of You", df_scaled, knn_model, n=10)
if recs_knn is not None:
    display(recs_knn.style.background_gradient(
        subset=["similarity_score"], cmap="Greens"))

In [ ]:
# ── KNN Evaluation ────────────────────────────────────────────
# Test on 5 popular songs — check if recommendations make sonic sense
print("KNN Evaluation — 3 recommendations per query")
print("=" * 60)
test_songs = df_scaled[df_scaled["popularity"] > 65].sample(5, random_state=42)
for _, song in test_songs.iterrows():
    recs = recommend_knn(song["name"], df_scaled, knn_model, n=3)
    if recs is not None:
        for i, row in recs.iterrows():
            print(f"    {i+1}. {row['name']} (sim: {row['similarity_score']:.4f})")

## 🔵 Model 2 — K-Means Clustering
**Task:** Group all songs into K clusters based on audio profile.

**How it works:** Assigns each song to the nearest centroid. Songs in the same cluster share a similar sound character.

**Input →** full dataset of audio feature vectors  
**Output →** cluster label per song, centroid fingerprints

In [ ]:
# ── Find optimal K ────────────────────────────────────────────
print("Finding optimal K... (~1-2 min)")

X_km = df_scaled[AUDIO_FEATURES].values
k_range = range(4, 16)
inertias, silhouettes = [], []

for k in k_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3)
    labels = km.fit_predict(X_km)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_km, labels, sample_size=8000, random_state=42)
    silhouettes.append(sil)
    print(f"  K={k:2d} | Inertia: {km.inertia_:>12,.0f} | Silhouette: {sil:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(list(k_range), inertias, "bo-")
ax1.set(title="Elbow Method", xlabel="K", ylabel="Inertia")
ax2.plot(list(k_range), silhouettes, "ro-")
ax2.set(title="Silhouette Score", xlabel="K", ylabel="Score")
plt.tight_layout()
plt.show()

BEST_K = list(k_range)[np.argmax(silhouettes)]
print(f"\n🏆 Best K = {BEST_K}")

In [ ]:
kmeans_model = MiniBatchKMeans(n_clusters=BEST_K, random_state=42, n_init=10)
df_clustered = df_scaled.copy()
df_clustered["kmeans_cluster"] = kmeans_model.fit_predict(X_km)

print("✅ K-Means trained!")
print("\nCluster sizes:")
print(df_clustered["kmeans_cluster"].value_counts().sort_index())

In [ ]:
# Cluster fingerprints heatmap
centers = pd.DataFrame(kmeans_model.cluster_centers_, columns=AUDIO_FEATURES)
plt.figure(figsize=(13, 5))
sns.heatmap(centers.T, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5,
            xticklabels=[f"C{i}" for i in range(BEST_K)])
plt.title("K-Means Cluster Audio Fingerprints", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# PCA 2D visualization
sample = df_clustered.sample(min(6000, len(df_clustered)), random_state=42)
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(sample[AUDIO_FEATURES].values)

plt.figure(figsize=(12, 7))
scatter = plt.scatter(coords[:, 0], coords[:, 1],
                      c=sample["kmeans_cluster"], cmap="tab20",
                      alpha=0.5, s=6)
plt.colorbar(scatter, label="Cluster")
plt.title(f"K-Means (K={BEST_K}) — PCA 2D", fontsize=13)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.tight_layout()
plt.show()

In [ ]:
def recommend_kmeans(song_name, df_clustered, n=10, popular_only=True):
    """
    Input  : song name
    Output : top-N popular songs from the same K-Means cluster
    """
    matches = df_clustered[
        df_clustered["name"].str.lower().str.contains(song_name.lower(), na=False)
    ]
    if matches.empty:
        print(f"❌ '{song_name}' not found."); return None

    song = matches.iloc[0]
    cid  = song["kmeans_cluster"]
    print(f"\n🎧 '{song['name']}' → Cluster {cid} "
          f"({(df_clustered['kmeans_cluster']==cid).sum():,} songs)")
    print("-" * 55)

    pool = df_clustered[
        (df_clustered["kmeans_cluster"] == cid) &
        (df_clustered["name"] != song["name"])
    ]
    if popular_only:
        pool = pool[pool["popularity"] > 0]

    return pool.sort_values("popularity", ascending=False).head(n)[
        ["name", "artists_clean", "year", "popularity", "kmeans_cluster"]
    ].reset_index(drop=True)


km_recs = recommend_kmeans("Shape of You", df_clustered, n=10)
if km_recs is not None:
    display(km_recs.style.background_gradient(
        subset=["popularity"], cmap="Purples"))

## 🌲 Model 3 — Random Forest (Popularity Prediction)
**Task:** Predict a song's popularity score (0–100) from its audio features.

**How it works:** Builds 200 decision trees on random feature subsets. The final prediction is the average of all trees. More robust than a single tree — handles non-linear relationships between audio features and popularity.

**Input →** 9 audio features  
**Output →** predicted popularity score (float, 0–100)

> ⚠️ Note: popularity has 27,892 zeros (obscure songs). We train on all data and track this in evaluation.

In [ ]:
# ── Train/test split ──────────────────────────────────────────
X_reg = df_scaled[AUDIO_FEATURES].values
y_reg = df_scaled[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

# ── Train Random Forest ───────────────────────────────────────
print("\nTraining Random Forest... (~1-2 min)")
rf_model = RandomForestRegressor(
    n_estimators=200,   # 200 trees
    max_depth=15,       # prevent overfitting
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("✅ Random Forest trained!")

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────
y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

print("📊 Random Forest Results:")
print(f"   RMSE : {rmse_rf:.4f}")
print(f"   MAE  : {mae_rf:.4f}")
print(f"   R²   : {r2_rf:.4f}")
print(f"\n   (Popularity is on 0–100 scale, but normalized to [0,1] here)")

In [ ]:
# ── Actual vs Predicted scatter ───────────────────────────────
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.15, s=5, color="steelblue")
plt.plot([0, 1], [0, 1], "r--", linewidth=2, label="Perfect prediction")
plt.title(f"Random Forest — Actual vs Predicted Popularity\nR² = {r2_rf:.4f}",
          fontsize=12)
plt.xlabel("Actual popularity (normalized)")
plt.ylabel("Predicted popularity")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────
importances = pd.Series(rf_model.feature_importances_, index=AUDIO_FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind="barh", color="steelblue")
plt.title("Random Forest — Feature Importances for Popularity", fontsize=12)
plt.xlabel("Importance score")
plt.tight_layout()
plt.show()

print("\nTop features driving popularity prediction:")
print(importances.sort_values(ascending=False).to_string())

In [ ]:
# ── Predict popularity for a new/hypothetical song ────────────
# You can change these values to predict popularity for any audio profile
new_song = pd.DataFrame([{
    "acousticness"    : 0.15,
    "danceability"    : 0.80,
    "energy"          : 0.75,
    "instrumentalness": 0.00,
    "liveness"        : 0.10,
    "loudness"        : -5.0,
    "speechiness"     : 0.05,
    "tempo"           : 120.0,
    "valence"         : 0.65,
}])

# Scale using the same scaler fitted on training data
new_song_scaled = scaler.transform(new_song[AUDIO_FEATURES])
pred_normalized = rf_model.predict(new_song_scaled)[0]

# Convert back to 0–100 scale
# popularity was NOT normalized by scaler (it's the target), so just multiply
pred_score = pred_normalized * 100
print(f"🎵 Predicted popularity: {pred_score:.1f} / 100")

## 📈 Model 4 — Linear Regression (Baseline Popularity Prediction)
**Task:** Same as Random Forest — predict popularity from audio features.

**Why include it:** Linear Regression is the baseline. Comparing its metrics with Random Forest shows how much the non-linear relationships matter. If LR performs similarly → the relationship is mostly linear. If RF >> LR → non-linearity matters a lot.

**Input →** 9 audio features  
**Output →** predicted popularity score

In [ ]:
# Uses same X_train, X_test, y_train, y_test from Model 3
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("✅ Linear Regression trained!")

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────
y_pred_lr = lr_model.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print("📊 Linear Regression Results:")
print(f"   RMSE : {rmse_lr:.4f}")
print(f"   MAE  : {mae_lr:.4f}")
print(f"   R²   : {r2_lr:.4f}")

In [ ]:
# ── Coefficients — which features push popularity up/down ─────
coef = pd.Series(lr_model.coef_, index=AUDIO_FEATURES).sort_values()

colors = ["coral" if c < 0 else "steelblue" for c in coef]
plt.figure(figsize=(9, 5))
plt.barh(coef.index, coef.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression Coefficients\n(blue=positive effect, red=negative)",
          fontsize=12)
plt.tight_layout()
plt.show()

print("\nCoefficients:")
print(coef.to_string())

In [ ]:
# ── RF vs LR comparison ───────────────────────────────────────
comparison = pd.DataFrame({
    "Metric" : ["RMSE", "MAE", "R²"],
    "Linear Regression" : [round(rmse_lr,4), round(mae_lr,4), round(r2_lr,4)],
    "Random Forest"     : [round(rmse_rf,4), round(mae_rf,4), round(r2_rf,4)],
}).set_index("Metric")

print("\n📊 Model Comparison (lower RMSE/MAE = better, higher R² = better):")
display(comparison.style.highlight_min(axis=1, subset=["RMSE","MAE"],
                                        color="lightgreen")
                         .highlight_max(axis=1, subset=["R²"],
                                        color="lightgreen"))

if r2_rf > r2_lr:
    print(f"\n→ Random Forest is better (R² {r2_rf:.4f} vs {r2_lr:.4f})")
    print("  Non-linear relationships exist between audio features & popularity.")
else:
    print("\n→ Linear Regression is competitive — relationship is mostly linear.")

## 🔴 Model 5 — DBSCAN (Density-Based Clustering)
**Task:** Find natural song groupings without pre-specifying K.

**How it works:** Groups points that are closely packed together. Points in low-density regions are labelled **-1 (noise/outliers)** — these are songs that don't fit any cluster (very unusual audio profiles).

**Key difference from K-Means:** Does not force every song into a cluster. Cluster shapes can be arbitrary, not just spherical.

**Input →** PCA-reduced audio features (2D, for speed)  
**Output →** cluster label per song (-1 = noise/outlier)

In [ ]:
# ── Reduce to 2D via PCA first (DBSCAN is slow on 9D, 160k rows) ──
print("Reducing to 2D with PCA for DBSCAN...")
X_full = df_scaled[AUDIO_FEATURES].values
pca_db = PCA(n_components=2, random_state=42)
X_2d   = pca_db.fit_transform(X_full)
print(f"Variance explained: "
      f"{pca_db.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# ── Tune eps using k-distance plot ────────────────────────────
# eps = the neighborhood radius. The k-distance plot helps pick it:
# look for the "elbow" — that's your eps.
from sklearn.neighbors import NearestNeighbors as NN

sample_idx = np.random.choice(len(X_2d), 10000, replace=False)
X_sample   = X_2d[sample_idx]

nn = NN(n_neighbors=5).fit(X_sample)
distances, _ = nn.kneighbors(X_sample)
k_distances  = np.sort(distances[:, -1])

plt.figure(figsize=(10, 4))
plt.plot(k_distances, color="steelblue")
plt.title("K-Distance Plot — Find eps (look for the elbow)", fontsize=12)
plt.xlabel("Points sorted by distance")
plt.ylabel("5th nearest neighbour distance")
plt.axhline(y=np.percentile(k_distances, 90), color="red",
            linestyle="--", label="90th percentile (suggested eps)")
plt.legend()
plt.tight_layout()
plt.show()

SUGGESTED_EPS = round(float(np.percentile(k_distances, 90)), 3)
print(f"\n💡 Suggested eps = {SUGGESTED_EPS}")
print("   You can adjust EPS below if clusters look too fragmented or merged.")

In [ ]:
# ── Train DBSCAN ──────────────────────────────────────────────
EPS      = SUGGESTED_EPS   # 👈 adjust if needed
MIN_SAMP = 10              # minimum points to form a cluster

print(f"Training DBSCAN (eps={EPS}, min_samples={MIN_SAMP})...")
print("This may take 2-3 minutes on the full dataset...")

dbscan_model = DBSCAN(eps=EPS, min_samples=MIN_SAMP, n_jobs=-1)
db_labels    = dbscan_model.fit_predict(X_2d)

df_clustered["dbscan_cluster"] = db_labels

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()

print(f"\n✅ DBSCAN done!")
print(f"   Clusters found  : {n_clusters}")
print(f"   Noise/outliers  : {n_noise:,} songs ({n_noise/len(db_labels)*100:.1f}%)")

In [ ]:
# ── Cluster size breakdown ────────────────────────────────────
cluster_counts = pd.Series(db_labels).value_counts().sort_index()
print("Cluster sizes (cluster_id : count):")
print(cluster_counts.to_string())

In [ ]:
# ── Visualize DBSCAN clusters ─────────────────────────────────
sample_idx_vis = np.random.choice(len(X_2d), 8000, replace=False)
coords_vis = X_2d[sample_idx_vis]
labels_vis = db_labels[sample_idx_vis]

# Separate noise from clusters
noise_mask   = labels_vis == -1
cluster_mask = labels_vis != -1

plt.figure(figsize=(12, 7))
plt.scatter(coords_vis[noise_mask, 0], coords_vis[noise_mask, 1],
            c="lightgrey", s=4, alpha=0.4, label="Noise/Outlier")
scatter = plt.scatter(coords_vis[cluster_mask, 0],
                      coords_vis[cluster_mask, 1],
                      c=labels_vis[cluster_mask],
                      cmap="tab20", s=6, alpha=0.6)
plt.colorbar(scatter, label="Cluster ID")
plt.title(f"DBSCAN — {n_clusters} Clusters + {n_noise:,} Noise Points",
          fontsize=13)
plt.xlabel(f"PC1 ({pca_db.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca_db.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Who are the outlier songs? ────────────────────────────────
# These are songs with very unusual audio profiles — don't fit any cluster
outliers = df_clustered[df_clustered["dbscan_cluster"] == -1]
print(f"🔍 Sample outlier songs (unusual audio profiles):")
display(outliers[["name", "artists_clean", "year", "popularity"] + AUDIO_FEATURES[:4]]
        .sort_values("popularity", ascending=False)
        .head(10))

In [ ]:
# ── DBSCAN vs K-Means comparison ─────────────────────────────
# Silhouette score on non-noise points only
non_noise_mask = db_labels != -1
if non_noise_mask.sum() > 1000:
    sil_db = silhouette_score(
        X_2d[non_noise_mask], db_labels[non_noise_mask],
        sample_size=8000, random_state=42
    )
    sil_km = silhouette_score(
        X_km, df_clustered["kmeans_cluster"].values,
        sample_size=8000, random_state=42
    )
    print("📊 Clustering Comparison:")
    print(f"   K-Means Silhouette : {sil_km:.4f}")
    print(f"   DBSCAN  Silhouette : {sil_db:.4f} (non-noise points only)")
    print(f"   DBSCAN  Noise      : {n_noise:,} songs flagged as outliers")

## 📋 Step 5: Full Model Summary

In [ ]:
print("=" * 65)
print("  FINAL MODEL SUMMARY")
print("=" * 65)

print(f"""
Model 1 — KNN (Content-Based)
  Task    : Recommend songs similar to a query song
  Metric  : Cosine similarity (higher = more similar)
  Usage   : recommend_knn("song name", df_scaled, knn_model)

Model 2 — K-Means Clustering
  Task    : Group songs into K sound-profile clusters
  K chosen: {BEST_K} (by silhouette score)
  Usage   : recommend_kmeans("song name", df_clustered)

Model 3 — Random Forest (Popularity Prediction)
  Task    : Predict popularity score from audio features
  RMSE    : {rmse_rf:.4f}  |  MAE : {mae_rf:.4f}  |  R² : {r2_rf:.4f}
  Trees   : 200, max_depth=15

Model 4 — Linear Regression (Baseline)
  Task    : Same as RF — baseline comparison
  RMSE    : {rmse_lr:.4f}  |  MAE : {mae_lr:.4f}  |  R² : {r2_lr:.4f}
  Winner  : {'Random Forest' if r2_rf > r2_lr else 'Linear Regression'}

Model 5 — DBSCAN
  Task    : Density-based clustering (no K needed)
  Clusters: {n_clusters}
  Noise   : {n_noise:,} songs flagged as outliers
""")
print("=" * 65)

## 💾 Step 6: Save All Models

In [ ]:
import shutil
from google.colab import files

os.makedirs("saved_models", exist_ok=True)

joblib.dump(knn_model,    "saved_models/knn_model.pkl")
joblib.dump(kmeans_model, "saved_models/kmeans_model.pkl")
joblib.dump(rf_model,     "saved_models/rf_model.pkl")
joblib.dump(lr_model,     "saved_models/lr_model.pkl")
joblib.dump(dbscan_model, "saved_models/dbscan_model.pkl")
joblib.dump(scaler,       "saved_models/scaler.pkl")
joblib.dump(pca_db,       "saved_models/pca_dbscan.pkl")
df_clustered.to_csv("saved_models/df_clustered.csv", index=False)

print("✅ Saved:", os.listdir("saved_models"))

shutil.make_archive("spotify_5_models", "zip", "saved_models")
files.download("spotify_5_models.zip")
print("✅ Download started!")

## 🎮 Step 7: Interactive Widget (All Models)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

song_input = widgets.Text(
    value="Shape of You",
    description="🎵 Song:",
    layout=widgets.Layout(width="420px")
)
model_select = widgets.RadioButtons(
    options=[
        "Model 1 — KNN (Audio Similarity)",
        "Model 2 — K-Means (Cluster)",
        "Model 3 — Random Forest (Predict Popularity)",
        "Model 4 — Linear Regression (Predict Popularity)",
        "Model 5 — DBSCAN (Find Outliers)",
    ],
    description="Model:",
    layout=widgets.Layout(width="500px")
)
popular_cb = widgets.Checkbox(value=True,
    description="Popular songs only (exclude popularity=0)")
n_slider = widgets.IntSlider(value=10, min=1, max=20, description="# Results:")
button   = widgets.Button(description="▶ Run", button_style="success",
                          layout=widgets.Layout(width="120px", height="36px"))
output   = widgets.Output()

def on_click(b):
    with output:
        clear_output(wait=True)
        song = song_input.value.strip()
        n    = n_slider.value
        pop  = popular_cb.value
        sel  = model_select.value

        if "KNN" in sel:
            recs = recommend_knn(song, df_scaled, knn_model, n=n, popular_only=pop)
            if recs is not None:
                display(recs.style.background_gradient(
                    subset=["similarity_score"], cmap="Greens"))

        elif "K-Means" in sel:
            recs = recommend_kmeans(song, df_clustered, n=n, popular_only=pop)
            if recs is not None:
                display(recs.style.background_gradient(
                    subset=["popularity"], cmap="Purples"))

        elif "Random Forest" in sel or "Linear" in sel:
            matches = df_scaled[df_scaled["name"].str.lower().str.contains(
                                song.lower(), na=False)]
            if matches.empty:
                print(f"❌ '{song}' not found.")
            else:
                s = matches.iloc[0]
                feats = s[AUDIO_FEATURES].values.reshape(1, -1)
                if "Random Forest" in sel:
                    pred = rf_model.predict(feats)[0] * 100
                    model_name = "Random Forest"
                else:
                    pred = lr_model.predict(feats)[0] * 100
                    model_name = "Linear Regression"
                print(f"🎧 '{s['name']}' by {s['artists_clean']}")
                print(f"   Actual popularity   : {s['popularity']}")
                print(f"   {model_name} predicted: {pred:.1f} / 100")

        elif "DBSCAN" in sel:
            matches = df_clustered[df_clustered["name"].str.lower().str.contains(
                                   song.lower(), na=False)]
            if matches.empty:
                print(f"❌ '{song}' not found.")
            else:
                s   = matches.iloc[0]
                cid = s["dbscan_cluster"]
                if cid == -1:
                    print(f"🎧 '{s['name']}' is flagged as a NOISE/OUTLIER song.")
                    print("   It has an unusual audio profile that doesn't fit any cluster.")
                else:
                    pool = df_clustered[
                        (df_clustered["dbscan_cluster"] == cid) &
                        (df_clustered["name"] != s["name"])
                    ]
                    if pop:
                        pool = pool[pool["popularity"] > 0]
                    recs = pool.sort_values("popularity", ascending=False).head(n)[
                        ["name", "artists_clean", "year",
                         "popularity", "dbscan_cluster"]
                    ].reset_index(drop=True)
                    print(f"🎧 '{s['name']}' → DBSCAN Cluster {cid}")
                    display(recs.style.background_gradient(
                        subset=["popularity"], cmap="Oranges"))

button.on_click(on_click)
display(widgets.VBox([
    song_input, model_select, popular_cb, n_slider, button, output
]))